Define the "Ban Date": 

Calculate the Baseline Date: 

Query Wayback: 

Scrape the Archive: 

In [3]:
import subprocess
subprocess.Popen(['caffeinate', '-t', '14400'])
print("Mac is caffeinated for the next 4 hours.")

Mac is caffeinated for the next 4 hours.


In [7]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from datetime import datetime, timedelta
import os

# Configuration
IDS_FILE = '/Users/vconklin24/Desktop/data/book_ids/goodreads_ids_temp.csv'
BANNED_FILE = '/Users/vconklin24/Desktop/data/datasets/pen_america_banned_books.csv'
OUTPUT_FILE = 'data/raw/goodreads_historical.csv'

def get_wayback_url(target_url, target_date):
    """Ask Wayback API for the snapshot closest to the target date."""
    timestamp = target_date.strftime("%Y%m%d")
    api_url = f"http://archive.org/wayback/available?url={target_url}&timestamp={timestamp}"
    try:
        response = requests.get(api_url, timeout=10).json()
        if response.get('archived_snapshots') and 'closest' in response['archived_snapshots']:
            return response['archived_snapshots']['closest']['url']
    except Exception as e:
        print(f"Wayback API error: {e}")
    return None

def scrape_historical_stats(archive_url):
    """Scrapes stats from the Wayback Machine's version of a Goodreads page."""
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) Chrome/119.0.0.0'}
    try:
        response = requests.get(archive_url, headers=headers, timeout=20)
        soup = BeautifulSoup(response.content, 'html.parser')
        
        # Historical pages often use older Goodreads layouts.
        # We try modern selectors first, then fallback to legacy ones.
        stats = {}

        # 1. Average Rating
        avg = soup.find("div", {"class": "RatingStatistics__rating"}) or soup.find("span", {"itemprop": "ratingValue"})
        stats['hist_avg_rating'] = avg.text.strip() if avg else None

        # 2. Rating Count
        count = soup.find("div", {"class": "RatingStatistics__meta"}) or soup.find("meta", {"itemprop": "ratingCount"})
        if count:
            # Handle meta tag or text tag
            val = count.get('content') if count.name == 'meta' else count.text
            stats['hist_rating_count'] = "".join(filter(str.isdigit, val)) if val else None

        return stats
    except Exception as e:
        print(f"Scrape error for {archive_url}: {e}")
        return None

def main():
    # Load Data
    ids_df = pd.read_csv(IDS_FILE)
    banned_df = pd.read_csv(BANNED_FILE)

    # Clean dates and find the EARLIEST ban date for each book
    banned_df['ban_date_dt'] = pd.to_datetime(banned_df['Date of Challenge/Removal'], errors='coerce')
    earliest_bans = banned_df.groupby(['Title', 'Author'])['ban_date_dt'].min().reset_index()

    # Merge to get URLs and Dates together
    merged_df = pd.merge(
        ids_df[ids_df['found'] == True], 
        earliest_bans, 
        left_on=['book_title', 'author'], 
        right_on=['Title', 'Author'], 
        how='inner'
    )

    results = []
    print(f"Starting historical scrape for {len(merged_df)} books...")

    for _, row in merged_df.iterrows():
        if pd.isna(row['ban_date_dt']):
            continue

        # Target: 3 months before ban
        target_date = row['ban_date_dt'] - timedelta(days=90)
        print(f"Processing: {row['book_title']} (Target Date: {target_date.date()})")

        archive_url = get_wayback_url(row['goodreads_url'], target_date)
        
        if archive_url:
            hist_stats = scrape_historical_stats(archive_url)
            if hist_stats:
                # Store data
                data = row.to_dict()
                data.update(hist_stats)
                data['archive_url_used'] = archive_url
                data['baseline_target_date'] = target_date.date()
                results.append(data)
        
        # Respectful delay
        time.sleep(3)

    # Save output
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)
    print(f"Success! Historical data saved to {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

/var/folders/6c/hjkys9j9037grwvjg6l8fl2m0000gq/T/ipykernel_15501/1215977108.py:58: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  banned_df['ban_date_dt'] = pd.to_datetime(banned_df['Date of Challenge/Removal'], errors='coerce')


Starting historical scrape for 256 books...
Processing: Nineteen Minutes (Target Date: 2024-04-02)
Processing: The Perks of Being a Wallflower (Target Date: 2024-05-03)
Processing: Beloved (Target Date: 2024-05-03)
Processing: Crank (Target Date: 2024-05-03)
Processing: Fallout (Target Date: 2024-05-03)
Processing: Glass (Target Date: 2024-05-03)
Processing: Identical (Target Date: 2024-04-02)
Processing: It's Your World - If You Don't Like It, Change It: Activism for Teenagers (Target Date: 2024-05-03)
Processing: Looking for Alaska (Target Date: 2024-05-03)
Processing: Melissa (George) (Target Date: 2024-05-03)
Processing: Nineteen Minutes (Target Date: 2024-04-02)
Processing: Redwood and Ponytail (Target Date: 2024-05-03)
Processing: Smoke (Target Date: 2024-05-03)
Processing: The Bluest Eye (Target Date: 2024-05-03)
Processing: The Hate U Give (Target Date: 2024-05-03)
Processing: The Kite Runner (Target Date: 2024-05-03)
Processing: The Perks of Being a Wallflower (Target Date: 20